<a href="https://colab.research.google.com/github/ranjith88697/Master-Thesis/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Data set collected from WITS(World Integrated Trade Solutions) HS Code level data from 2005 - 2025

In [2]:
import pandas as pd

trains = pd.read_csv("/content/Data_dataallcountry.csv")
comtrade = pd.read_csv("/content/Data_comtradealldata.csv", encoding='latin1')

In [3]:
trains.head()

,Partner,HS code,Product Name,Year,TariffRate
0,Djibouti,39,PLASTICS AND ARTICLES THEREOF,2005,29.37
1,Djibouti,87,VEHICLES OTHER THAN RAILWAY OR TRAMWAY ROLLING...,2005,25.51
2,"Gambia, The",39,PLASTICS AND ARTICLES THEREOF,2009,20.00
3,"Gambia, The",85,ELECTRICAL MACHINERY AND EQUIPMENT AND PARTS T...,2008,18.90
4,"Gambia, The",94,"FURNITURE; BEDDING, MATTRESSES, MATTRESS SUPPO...",2007,20.00


In [4]:
comtrade.head()

,HS code,Partner,Year,ImportValue
0,84,Anguila,2024,111.842
1,84,Albania,2005,204.266
2,84,Albania,2006,106.948
3,84,Albania,2007,99.151
4,84,Albania,2008,49.058


In [5]:
merged = pd.merge(
    trains,
    comtrade,
    on=["Year", "HS code", "Partner"],
    how="inner"
)


In [7]:
merged.head()

,Partner,HS code,Product Name,Year,TariffRate,ImportValue
0,Djibouti,39,PLASTICS AND ARTICLES THEREOF,2005,29.37,0.436
1,"Gambia, The",85,ELECTRICAL MACHINERY AND EQUIPMENT AND PARTS T...,2008,18.90,232.621
2,Antigua and Barbuda,39,PLASTICS AND ARTICLES THEREOF,2005,14.17,0.375
3,Antigua and Barbuda,84,"NUCLEAR REACTORS, BOILERS, MACHINERY AND MECHA...",2009,8.66,343.192
4,Antigua and Barbuda,87,VEHICLES OTHER THAN RAILWAY OR TRAMWAY ROLLING...,2005,18.15,6.029


In [6]:
merged.shape
merged.isna().sum()
#merged.head()


,0
Partner,0
HS code,0
Product Name,0
Year,0
TariffRate,14
ImportValue,0


In [8]:
merged = merged.sort_values(["Partner", "HS code", "Year"])
merged["ImportValue_lag1"] = merged.groupby(["Partner", "HS code"])["ImportValue"].shift(1)


In [9]:
merged["ImportGrowth"] = (
    merged["ImportValue"] - merged["ImportValue_lag1"]
) / merged["ImportValue_lag1"]


In [12]:
trains.head()

,Partner,HS code,Product Name,Year,TariffRate
0,Djibouti,39,PLASTICS AND ARTICLES THEREOF,2005,29.37
1,Djibouti,87,VEHICLES OTHER THAN RAILWAY OR TRAMWAY ROLLING...,2005,25.51
2,"Gambia, The",39,PLASTICS AND ARTICLES THEREOF,2009,20.00
3,"Gambia, The",85,ELECTRICAL MACHINERY AND EQUIPMENT AND PARTS T...,2008,18.90
4,"Gambia, The",94,"FURNITURE; BEDDING, MATTRESSES, MATTRESS SUPPO...",2007,20.00


In [20]:
comtrade.head()

,HS code,Partner,Year,ImportValue
0,84,Anguila,2024,111.842
1,84,Albania,2005,204.266
2,84,Albania,2006,106.948
3,84,Albania,2007,99.151
4,84,Albania,2008,49.058


In [19]:
merged.head()

,Partner,HS code,Product Name,Year,TariffRate,ImportValue,ImportValue_lag1,ImportGrowth
483,Afghanistan,39,PLASTICS AND ARTICLES THEREOF,2013,5.31,4.325,NaN,NaN
4555,Afghanistan,39,PLASTICS AND ARTICLES THEREOF,2018,3.05,37.132,4.325,7.585434
4406,Afghanistan,73,ARTICLES OF IRON OR STEEL,2008,2.50,15.738,NaN,NaN
3230,Afghanistan,73,ARTICLES OF IRON OR STEEL,2012,2.50,21.145,15.738,0.343563
12161,Afghanistan,73,ARTICLES OF IRON OR STEEL,2013,2.50,30.883,21.145,0.460534


In [10]:
merged.to_csv("hs_level_data.csv", index=False)

In [11]:
trains.columns
comtrade.columns
merged.columns


Index(['Partner', 'HS code', 'Product Name', 'Year', 'TariffRate',
       'ImportValue', 'ImportValue_lag1', 'ImportGrowth'],
      dtype='object')

In [13]:
import pandas as pd

# Load WTO summary table
df = pd.read_excel("/content/Summ_all_EN_WTP25.xlsx", sheet_name=0)

# Rename columns based on your screenshot
df = df.rename(columns={
    "Country/Territory": "Partner",
    "Year of MFN applied tariff": "Year",
    "Simple average - MFN applied": "MFN_Simple",
    "Duty-free - MFN applied": "MFN_DutyFreeShare",
    "Maximum duty - MFN applied": "MFN_MaxRate"
})

# Compute dutiable share
df["MFN_DutiableShare"] = 100 - df["MFN_DutyFreeShare"]

# Keep only needed columns
df = df[["Partner", "Year", "MFN_Simple", "MFN_DutyFreeShare", "MFN_DutiableShare", "MFN_MaxRate"]]

# Create full panel 2005–2025
years = list(range(2005, 2026))
countries = df["Partner"].unique()

panel = pd.MultiIndex.from_product([countries, years], names=["Partner", "Year"])
panel_df = pd.DataFrame(index=panel).reset_index()

# Merge WTO data (static values)
macro_wto = panel_df.merge(df.drop(columns=["Year"]), on="Partner", how="left")

# Save
macro_wto.to_csv("macro_wto_static.csv", index=False)

print(macro_wto.head())
print(macro_wto.tail())


                                             Partner  Year  MFN_Simple  \
0  Afghanistan                                   ...  2005         NaN   
1  Afghanistan                                   ...  2006         NaN   
2  Afghanistan                                   ...  2007         NaN   
3  Afghanistan                                   ...  2008         NaN   
4  Afghanistan                                   ...  2009         NaN   

   MFN_DutyFreeShare  MFN_DutiableShare MFN_MaxRate  
0                NaN                NaN         NaN  
1                NaN                NaN         NaN  
2                NaN                NaN         NaN  
3                NaN                NaN         NaN  
4                NaN                NaN         NaN  
                                                Partner  Year  MFN_Simple  \
3208  Zimbabwe                                      ...  2021         NaN   
3209  Zimbabwe                                      ...  2022         NaN   


In [14]:
import pandas as pd

# -----------------------------
# WTO MFN PANEL
# -----------------------------
wto = pd.read_csv("macro_wto_static.csv")

# -----------------------------
# COUNTRY-LEVEL GDP GROWTH
# -----------------------------
country = pd.read_csv(
    "growth_wdi_country.csv",
    encoding="utf-8-sig",
    engine="python"
)

# Filter to GDP growth indicator
country = country[country["Indicator Code"] == "NY.GDP.MKTP.KD.ZG"]

# Reshape from wide to long format
country_long = country.melt(
    id_vars=["Country Name"],
    var_name="Year",
    value_name="CountryGrowth"
)

# Clean and filter
country_long = country_long.rename(columns={"Country Name": "Partner"})
country_long["Year"] = pd.to_numeric(country_long["Year"], errors="coerce")
country_long = country_long.dropna(subset=["Year"])
country_long = country_long[country_long["Year"].between(2005, 2025)]

# -----------------------------
# WORLD-LEVEL GDP GROWTH
# -----------------------------
world = pd.read_csv("growth_wdi_world_clean.csv")
world["Year"] = pd.to_numeric(world["Year"], errors="coerce")

wto["Partner"] = wto["Partner"].str.strip()
country_long["Partner"] = country_long["Partner"].str.strip()


# -----------------------------
# MERGE ALL THREE DATASETS
# -----------------------------
macro = (
    wto
    .merge(country_long, on=["Partner", "Year"], how="left")
    .merge(world, on="Year", how="left")
)

# Save final macro dataset
macro.to_csv("macro.csv", index=False)

print(macro.head())
print(macro.tail())


       Partner  Year  MFN_Simple  MFN_DutyFreeShare  MFN_DutiableShare  \
0  Afghanistan  2005         NaN                NaN                NaN   
1  Afghanistan  2006         NaN                NaN                NaN   
2  Afghanistan  2007         NaN                NaN                NaN   
3  Afghanistan  2008         NaN                NaN                NaN   
4  Afghanistan  2009         NaN                NaN                NaN   

  MFN_MaxRate CountryGrowth  WorldGrowth  
0         NaN     11.229715     4.054536  
1         NaN      5.357403     4.476170  
2         NaN      13.82632     4.413845  
3         NaN      3.924984     2.092395  
4         NaN     21.390528    -1.318649  
       Partner  Year  MFN_Simple  MFN_DutyFreeShare  MFN_DutiableShare  \
3208  Zimbabwe  2021         NaN                NaN                NaN   
3209  Zimbabwe  2022         NaN                NaN                NaN   
3210  Zimbabwe  2023         NaN                NaN                NaN   
3

In [15]:
country_long.head()

,Partner,Year,CountryGrowth
12768,Aruba,2005.0,-0.383138
12769,Africa Eastern and Southern,2005.0,6.173659
12770,Afghanistan,2005.0,11.229715
12771,Africa Western and Central,2005.0,5.797434
12772,Angola,2005.0,14.154778


In [16]:
import pandas as pd

# Load HS-level dataset
hs = pd.read_csv("hs_level_data.csv")

# Load macro dataset
macro = pd.read_csv("macro.csv")

# Clean partner names (important!)
hs["Partner"] = hs["Partner"].str.strip()
macro["Partner"] = macro["Partner"].str.strip()

# Merge
merged_final = hs.merge(
    macro,
    on=["Partner", "Year"],
    how="left"
)

# Save final dataset
merged_final.to_csv("final_model_data.csv", index=False)

merged_final.head()


,Partner,HS code,Product Name,Year,TariffRate,ImportValue,ImportValue_lag1,ImportGrowth,MFN_Simple,MFN_DutyFreeShare,MFN_DutiableShare,MFN_MaxRate,CountryGrowth,WorldGrowth
0,Afghanistan,39,PLASTICS AND ARTICLES THEREOF,2013,5.31,4.325,NaN,NaN,NaN,NaN,NaN,NaN,5.600745,2.888142
1,Afghanistan,39,PLASTICS AND ARTICLES THEREOF,2018,3.05,37.132,4.325,7.585434,NaN,NaN,NaN,NaN,1.189228,3.273958
2,Afghanistan,73,ARTICLES OF IRON OR STEEL,2008,2.50,15.738,NaN,NaN,NaN,NaN,NaN,NaN,3.924984,2.092395
3,Afghanistan,73,ARTICLES OF IRON OR STEEL,2012,2.50,21.145,15.738,0.343563,NaN,NaN,NaN,NaN,12.752287,2.731210
4,Afghanistan,73,ARTICLES OF IRON OR STEEL,2013,2.50,30.883,21.145,0.460534,NaN,NaN,NaN,NaN,5.600745,2.888142


In [17]:
import pandas as pd

df = merged_final.copy()

# Sort for lag creation
df = df.sort_values(["Partner", "HS code", "Year"])

# Create lagged tariff (tariff persistence)
df["TariffRate_lag1"] = df.groupby(["Partner", "HS code"])["TariffRate"].shift(1)

# Drop rows where current tariff is missing
df = df.dropna(subset=["TariffRate"])
